

## Step 1: Installs

In [1]:
%pip install -q langchain chromadb tiktoken langchain-community langchain-mistralai langchain-chroma python-dotenv pypdf

Note: you may need to restart the kernel to use updated packages.


## Step 2: Imports and Environment Setup

In [2]:
import os
from pathlib import Path

from dotenv import load_dotenv
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.document_loaders import PyPDFLoader
from langchain_mistralai import MistralAIEmbeddings, ChatMistralAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_community.vectorstores import Chroma


load_dotenv()

MISTRAL_API_KEY = os.getenv("MISTRAL_API_KEY")
if not MISTRAL_API_KEY:
    print("WARNING: MISTRAL_API_KEY is not set in .env. Set it before running embedding/LLM cells.")



## Step 3: Getting PDF Files From working Directory And Add Company_id

In [3]:
#add which file to load with company id
Tenant_PDF_map_key = {
    "TechNova": ["technova_rag_benchmark.pdf"],
    "GreenLeaf": ["greenleaf_rag_benchmark.pdf"],
    "Finesecure": ["finsecure_rag_benchmark.pdf"],
}

if not Tenant_PDF_map_key:
    raise ValueError("Tenant_PDF_map_KEY is empty")

Tenant_PDF_map = {}
for company_id, file_names in Tenant_PDF_map_key.items():
    if not company_id or not str(company_id).strip():
        raise ValueError("invalid company_id found. Company IDs must be non-empty strings.")
    if not file_names:
        raise ValueError(f"No PDF files for company: {company_id}")

    #checking if file exists and is a pdf and adding to map
    for file_name in file_names:
        pdf_path = Path(file_name)
        if not pdf_path.exists():
            raise FileNotFoundError(f"PDF not found: {file_name}")
        if pdf_path.suffix.lower() != ".pdf":
            raise ValueError(f"file is not a PDF: {file_name}")
        Tenant_PDF_map.setdefault(company_id, []).append(pdf_path)

allowed_companies = set(Tenant_PDF_map.keys())

print("Configured tenant to PDF mapping:")
for company, files in sorted(Tenant_PDF_map.items()):
    print(f"- {company}: {[f.name for f in files]}")

print(f"\nTotal file found: {len(allowed_companies)}")

Configured tenant to PDF mapping:
- Finesecure: ['finsecure_rag_benchmark.pdf']
- GreenLeaf: ['greenleaf_rag_benchmark.pdf']
- TechNova: ['technova_rag_benchmark.pdf']

Total file found: 3


## Step 4: PDF Chunking And Adding Metadata

In [4]:

def load_and_tag_pdf_documents(file_path: Path, company_id: str):
    """Load PDF and attach metadata to each extracted page."""

    loader = PyPDFLoader(str(file_path))
    docs = loader.load()
    for d in docs:
        d.metadata.update({
            "company_id": company_id,
            "source": file_path.name,
        })
    return docs

all_raw_docs = []
for company_id, files in Tenant_PDF_map.items():
    for file_path in files:
        all_raw_docs.extend(load_and_tag_pdf_documents(file_path, company_id))

if not all_raw_docs:
    raise ValueError("No PDF content was loaded. Check your PDF files and mapping.")

splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=50)
chunked_docs = splitter.split_documents(all_raw_docs)

print(f"Raw PDF pages loaded: {len(all_raw_docs)}")
print(f"Chunks created: {len(chunked_docs)}")
print("Sample metadata from first chunk:")
print(chunked_docs[0].metadata)

Raw PDF pages loaded: 81
Chunks created: 438
Sample metadata from first chunk:
{'producer': 'ReportLab PDF Library - (opensource)', 'creator': '(unspecified)', 'creationdate': '2026-03-12T06:02:33+00:00', 'author': '(anonymous)', 'keywords': '', 'moddate': '2026-03-12T06:02:33+00:00', 'subject': '(unspecified)', 'title': '(anonymous)', 'trapped': '/False', 'source': 'technova_rag_benchmark.pdf', 'total_pages': 27, 'page': 0, 'page_label': '1', 'company_id': 'TechNova'}


## Step 5: Vector Database Creation

In [5]:
DB_DIR = Path("chroma_db")

if not MISTRAL_API_KEY:
    raise EnvironmentError("MISTRAL_API_KEY is required to embed and query documents.")

embeddings = MistralAIEmbeddings(model="mistral-embed")

vectorstore = Chroma(
    collection_name="enterprise_policies",
    embedding_function=embeddings,
    persist_directory=str(DB_DIR)
)

# Reset vectordb (delete): delete old collection contents.
try:
    vectorstore.delete_collection()
except Exception:
    pass

vectorstore = Chroma(
    collection_name="enterprise_policies",
    embedding_function=embeddings,
    persist_directory=str(DB_DIR)
)

vectorstore.add_documents(chunked_docs)

try:
    vectorstore.persist()
except Exception:
    # Newer versions auto-persist; keep compatibility across versions.
    pass

print("ChromaDB collection is ready with metadata-tagged chunks.")

c:\Users\rishabh singh\Desktop\knowledge-enterprise-automantion\myvenv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
C:\Users\rishabh singh\AppData\Local\Temp\ipykernel_11532\3972465859.py:8: LangChainDeprecationWarning: The class `Chroma` was deprecated in LangChain 0.2.9 and will be removed in 1.0. An updated version of the class exists in the `langchain-chroma package and should be used instead. To use it run `pip install -U `langchain-chroma` and import as `from `langchain_chroma import Chroma``.
  vectorstore = Chroma(


ChromaDB collection is ready with metadata-tagged chunks.


C:\Users\rishabh singh\AppData\Local\Temp\ipykernel_11532\3972465859.py:29: LangChainDeprecationWarning: Since Chroma 0.4.x the manual persistence method is no longer supported as docs are automatically persisted.
  vectorstore.persist()


## Step 6:Retrieval Chain

In [6]:
SECURE_PROMPT = ChatPromptTemplate.from_template(
    """You are a secure enterprise policy assistant.
Answer the user's question using ONLY the context below.
If the answer is not explicitly present in the context, reply exactly: I don't know
Do not infer, guess, or use external knowledge.

Tenant Context:
{context}

Question: {question}
Answer:"""
)

llm = ChatMistralAI(model="mistral-small-latest", temperature=0)
parser = StrOutputParser()

def secure_chat(query: str, user_company_id: str) -> str:
    """tenant-aware RAG chat function with metadata filtering."""
    try:
        if not query or not query.strip():
            return "I don't know"

        if user_company_id not in allowed_companies:
            return "I don't know"

        retriever = vectorstore.as_retriever(
            search_type="similarity",
            search_kwargs={
                "k": 4,
                "filter": {"company_id": user_company_id}
            }
        )

        docs = retriever.invoke(query)
        if not docs:
            return "I don't know"

        context_text = "\n\n".join([d.page_content for d in docs])
        chain = SECURE_PROMPT | llm | parser
        response = chain.invoke({"context": context_text, "question": query}).strip()

        #if model returns empty text, do not guess.
        return response if response else "I don't know"
    except Exception as exc:
        print(f"[secure_chat error] {exc}")
        return "I don't know"

## Step 7: Testing

In [17]:
# Display available tenants
available_companies = sorted(allowed_companies)
print(f"Available tenants: {available_companies}\n")

# ask query with company_id
query = " Which policy section governs termination procedures"
company_id = "Finesecure"

if not query:
    print("Error: Query cannot be empty.")
elif company_id not in available_companies:
    print(f"Error: Company '{company_id}' not found in available tenants.")
else:
    print("=" * 80)
    print(f"Query: {query}")
    print(f"Company: {company_id}")
    answer = secure_chat(query, company_id)
    print(f"Answer: {answer}")


Available tenants: ['Finesecure', 'GreenLeaf', 'TechNova']

Query:  Which policy section governs termination procedures
Company: Finesecure
Answer: 14. Termination and Exit Policy


In [8]:

# What are the official working hours for employees?
# • Which section defines disciplinary actions for policy violations?
# • According to Clause DS.3, what rules apply to confidential data handling?
# • How does the attendance policy interact with disciplinary actions?
# • Which departments follow rotational shifts?
# • What section references employee benefits?
# • How many clauses exist within the Remote Work Policy section?
# • What cross referenced section discusses confidentiality obligations?
# • What are the standard break durations listed in the working hours table?
# • Which policy section governs termination procedures?
